In [1]:

import pandas as pd
import numpy as np
from numpy.linalg import norm

df = pd.read_csv('course_ratings.csv')

rating_matrix = df.pivot(index='userId', columns='courseId', values='rating').fillna(0)

def cosine_similarity(v1, v2):
    if norm(v1) == 0 or norm(v2) == 0:
        return 0
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

course_titles = {
    "cm8u9l665000csprox9w7z2f9": "Web Development 101",
    "cm8u9l65y000asproacv1c3c7": "Python for Beginners",
    "cm8u9l65r0008spro8d81clgm": "Data Science Essentials",
    "cm8u9l65e0006sproli0k8mua": "UI/UX Design Basics"
}

recommendations = {}

for target_user in rating_matrix.index:
    target_vector = rating_matrix.loc[target_user].values

    similarities = {}
    for other_user in rating_matrix.index:
        if other_user != target_user:
            sim = cosine_similarity(target_vector, rating_matrix.loc[other_user].values)
            similarities[other_user] = sim

    similar_users = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

    course_scores = {}
    for other_user, similarity in similar_users:
        for course in rating_matrix.columns:
            if rating_matrix.loc[target_user, course] == 0:  # course not rated by target user
                course_scores.setdefault(course, 0)
                course_scores[course] += similarity * rating_matrix.loc[other_user, course]

    top_courses = sorted(course_scores.items(), key=lambda x: x[1], reverse=True)[:4]
    recommendations[target_user] = top_courses

print("\n📚 Course Recommendations:")
for user_id, recs in recommendations.items():
    print(f"\n👤 User {user_id}:")
    if recs:
        for course_id, score in recs:
            title = course_titles.get(course_id, course_id)  # fallback to course_id if title not found
            print(f"  - {title} (score: {score:.2f})")
    else:
        print("  No recommendations available.")



📚 Course Recommendations:

👤 User cm8u9l61c0000sproker36xb0:
  - Web Development 101 (score: 1.95)
  - Python for Beginners (score: 1.87)

👤 User cm8u9l64j0001sprodjtgvjlm:
  - Data Science Essentials (score: 2.50)
  - Web Development 101 (score: 0.00)

👤 User cm8u9l64q0002spro0bwxut2c:
  - UI/UX Design Basics (score: 1.95)
  - Python for Beginners (score: 0.00)
